In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import os
import time
import multiprocessing as mp
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,f1_score,recall_score,roc_auc_score
from itertools import product


In [ ]:
# Descargar el conjunto de datos completo
# Esto devolverá la ruta del directorio local donde se ha descargado el conjunto de datos.
dataset_root_path = kagglehub.dataset_download(
    "meowmeowmeowmeowmeow/gtsrb-german-traffic-sign"
)

# Listar todos los directorios dentro de la carpeta descargada para asegurar que estén completos
print("\nDirectorios en el conjunto de datos:")
for root, dirs, files in os.walk(dataset_root_path):
    # Imprimir directorios
    for name in dirs:
        print(os.path.join(root, name) + '/')


Using Colab cache for faster access to the 'gtsrb-german-traffic-sign' dataset.

Directorios en el conjunto de datos:
/kaggle/input/gtsrb-german-traffic-sign/Meta
/kaggle/input/gtsrb-german-traffic-sign/meta
/kaggle/input/gtsrb-german-traffic-sign/Test
/kaggle/input/gtsrb-german-traffic-sign/test
/kaggle/input/gtsrb-german-traffic-sign/Train
/kaggle/input/gtsrb-german-traffic-sign/train
/kaggle/input/gtsrb-german-traffic-sign/Train/7
/kaggle/input/gtsrb-german-traffic-sign/Train/17
/kaggle/input/gtsrb-german-traffic-sign/Train/19
/kaggle/input/gtsrb-german-traffic-sign/Train/22
/kaggle/input/gtsrb-german-traffic-sign/Train/2
/kaggle/input/gtsrb-german-traffic-sign/Train/35
/kaggle/input/gtsrb-german-traffic-sign/Train/23
/kaggle/input/gtsrb-german-traffic-sign/Train/10
/kaggle/input/gtsrb-german-traffic-sign/Train/5
/kaggle/input/gtsrb-german-traffic-sign/Train/36
/kaggle/input/gtsrb-german-traffic-sign/Train/20
/kaggle/input/gtsrb-german-traffic-sign/Train/27
/kaggle/input/gtsrb-germa

#Reducción de datos
Vamos a reducir el dataset original de a 30k imagenes

In [ ]:
import pandas as pd

train_csv = os.path.join(dataset_root_path, "Train.csv")

df = pd.read_csv(train_csv)

print(df.head())
print(df.shape)

   Width  Height  Roi.X1  Roi.Y1  Roi.X2  Roi.Y2  ClassId  \
0     27      26       5       5      22      20       20   
1     28      27       5       6      23      22       20   
2     29      26       6       5      24      21       20   
3     28      27       5       6      23      22       20   
4     28      26       5       5      23      21       20   

                             Path  
0  Train/20/00020_00000_00000.png  
1  Train/20/00020_00000_00001.png  
2  Train/20/00020_00000_00002.png  
3  Train/20/00020_00000_00003.png  
4  Train/20/00020_00000_00004.png  
(39209, 8)


In [ ]:
df_muestra = df.sample(n=30000, random_state=42)

print(df_muestra.shape)

(30000, 8)


## Preprocesamiento

In [ ]:
from PIL import Image

X = []
y = []

#iteramos sobre el df para cargar, redimensionar y coleccionar las imagenes y etiquetas
for index, row in df_muestra.iterrows():
    # construimos la ruta completa hacia las imagenes
    image_path = os.path.join(dataset_root_path, row['Path'])

    try:
        #Cargar la imagen
        img = Image.open(image_path)
        img = img.resize((96, 96)) #primero fue de 32px-> 64px, ->96px
        #Convertimos la imagen a un arreglo para integrarla a X
        X.append(np.array(img))
        #Integramos ClassId a y
        y.append(row['ClassId'])
    except Exception as e:
        print(f"Error procesando {image_path}: {e}")

# Convertir las listas a arreglos
X = np.array(X)
y = np.array(y)

print(f"Dimensiones de imagenes procesadas (X): {X.shape}")
print(f"Dimensiones de etiquetas (y): {y.shape}")

Dimensiones de imagenes procesadas (X): (30000, 96, 96, 3)
Dimensiones de etiquetas (y): (30000,)


In [ ]:
import cv2
#aplicaremos filtro gaussiano a cada imagen en X. Inicialmente se aplicó (5,5) pero no dio buenos resultados, reducimos a (3,3)
#El valor 0 indica que la desviación estándar en las direcciones X e Y se calcula a partir del tamaño del kernel
X_smoothed = np.array([cv2.GaussianBlur(img, (3, 3), 0) for img in X])

print(f"dimensiones de imagenes suavizadas(X_smoothed): {X_smoothed.shape}")

dimensiones de imagenes suavizadas(X_smoothed): (30000, 96, 96, 3)


### SIFT

Es más comun realizar sift sobre grises pero considerando que el color es importante para este conjunto ya que colores como amarillo, rojos y blancos son de importancia visual dejaremos esta configuracion. Inicialmente sí se realizó sobre grises pero no dio buenos resultados respecto al accuracy


In [ ]:
import cv2

#inicializar sift
sift = cv2.SIFT_create()

all_descriptors = []

for i, img_gray in enumerate(X_smoothed):
    # Detectar keypoints y descriptores
    keypoints, descriptors = sift.detectAndCompute(img_gray, None)

    if descriptors is not None:
        all_descriptors.append(descriptors)

# Concatenar todos los descriptores en un único array NumPy
# Este array se utilizará para entrenar el vocabulario BOVW

if all_descriptors:
    all_descriptors_np = np.vstack(all_descriptors)
    print(f"Total SIFT descriptores extraídos: {all_descriptors_np.shape}")
else:
    all_descriptors_np = np.array([])
    print("No se extrayeron descriptores SIFT.")

Total SIFT descriptores extraídos: (943903, 128)


# Flujos de clasificadores
A partir de este punto se implementan los clasificadores SVM y Random Forest con su correspondiente flujo. Verifique los archivos anexos dentro de este repositorio